# 01 · Define & Explore — di-zinc target prep, active-site-rim hotspots, binder + occlusion metrics

**Standard slot:** *define & explore.* **For Project 10 this means:** clean **NDM-1** while
**preserving both catalytic Zn²⁺ ions**, select the **hotspot residues on the active-site rim** (the
walls of the substrate-access channel — the occluding epitope), write down the binder + occlusion
metrics + cutoffs, and run a deterministic **mock** mini-run as your "hello-world" (D0).

> **Defensive anti-AMR framing.** The goal is to **inhibit** NDM-1 so a last-resort antibiotic works
> again — *not* to enhance resistance or pathogen fitness. The binder occludes the substrate channel;
> it never stabilizes or protects the enzyme. See the Responsible Research sections of `README.md` /
> `MANUAL.md`.

Run `00_setup.ipynb` first in this session. A real binder campaign wants an **A100** (see
`MANUAL.md §2`); everything here runs on a no-GPU **mock** backend so you can build the plumbing
anywhere, then switch to the real backend on Colab Pro / A100.

## The binder + occlusion metrics, precisely

| Metric | Range | Means | Does **not** mean |
|--------|-------|-------|-------------------|
| pLDDT | 0–100 | per-residue *local* confidence of the binder | thermostability / ΔG |
| **pae_interaction** | Å | AF2-Multimer error across the **binder–target interface** (the key binder metric) | measured affinity |
| scRMSD | Å | designed-vs-predicted Cα-RMSD (self-consistency) | binding/function |
| rosetta_dG | REU | interface energy (more negative = stronger) | a guarantee it binds |
| shape complementarity | 0–1 | interface packing quality | epitope correctness |
| **occlusion** | 0–1 | fraction of the substrate-access channel blocked (mechanism) | **inhibition** (needs the kinetics assay) |
| **specificity** | 0–1 | NDM-1-selectivity vs human metalloenzymes (higher = safer) | proof of no off-target effect |
| TM-score | 0–1 | similarity to nearest known fold (<0.5 ≈ novel) | a pass/fail of correctness |

The shared `"binder"` cutoffs: **scRMSD ≤ 2.5, pLDDT ≥ 80, pae_interaction ≤ 10, rosetta_dG ≤ −30,
sc ≥ 0.6.** Project additions (notebook 04): **occlusion ≥ 0.5, specificity ≥ 0.5.** `pae_interaction`
is the single most important binder metric — but a low value is *confidence*, **not** affinity. And
**binding ≠ inhibition**: a passing, occluding design is a **hypothesis** until the nitrocefin /
carbapenem IC50 assay (notebook 05).

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Di-zinc target prep + active-site-rim hotspots

The design target is **NDM-1 with its di-zinc active site preserved**, and the hotspots are the
**active-site-rim** residues that wall the **substrate (carbapenem) access channel** — steering the
binder there is what makes it an *occluding inhibitor*, not just a sticker. Fetch the candidate
structures with `data/download_data.py` (3SPU / 4EYL — **verify on RCSB**), isolate the NDM-1 chain,
remove waters/buffer and any hydrolyzed-substrate ligand, **keep both Zn²⁺ ions as heteroatoms**, and
read the rim residues off the channel.

> **Do NOT** strip the Zn²⁺ ions or design over the Zn-coordinating His/Cys/Asp residues — the binder
> sits on the rim and occludes substrate access; it does not replace the metal ligands.

Below we just *declare* an EXAMPLE rim-hotspot set so the notebook runs end-to-end; **replace it with
the residues you derive from the actual di-zinc structure** (numbering depends on the PDB you verify).

In [ ]:
import binder_tools as bt

TARGET = "NDM1"                      # cleaned NDM-1 with BOTH Zn2+ preserved (you produce this from 3SPU/4EYL)
# EXAMPLE active-site-rim hotspots — VERIFY/REPLACE from the cleaned di-zinc structure (data/README.md).
# These are placeholders so the plumbing runs; real numbering depends on the PDB chain you clean.
# They name the RIM that walls the substrate-access channel — NOT the Zn-coordinating ligand residues.
HOTSPOTS = bt.parse_hotspots("A120,A220,A228")   # EXAMPLE_DATA placeholder rim residues
print("target  :", TARGET, "(di-zinc active site — both Zn2+ preserved)")
print("hotspots:", HOTSPOTS, " (EXAMPLE active-site-rim residues — replace with your verified rim)")

## 2 · Mock hello-world: a tiny two-paradigm mini-run

`scripts/binder_tools.py` exposes both paradigms behind one API:
`generate_binders_bindcraft(...)` and `generate_binders_rfdiffusion(...)` (the latter → **LigandMPNN**,
Zn-aware, on the real backend), plus `af2_multimer(...)` (the scorer). The **mock** backend is
deterministic and GPU-free so you can develop the plumbing. **Never report mock numbers as real** —
they are `SYNTHETIC` by construction, and there are **no fabricated IC50s** anywhere.

In [ ]:
# A few designs from each paradigm, scored by mock AF2-Multimer. All numbers are SYNTHETIC.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=3, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=3, tool="mock")
bt.score_designs(bc, tool="mock")
bt.score_designs(rf, tool="mock")

d = bc[0]
print("example BindCraft design:")
print("  id   :", d.design_id)
print("  len  :", d.length, "aa")
print("  seq  :", d.sequence)
print("  pae_interaction =", d.pae_interaction, " scrmsd =", d.scrmsd,
      " sc =", d.shape_complementarity, " (SYNTHETIC)")
print("  synthetic flag  :", d.synthetic, "->", d.notes[0])
print("\nReminder: switch tool='mock' -> 'bindcraft'/'rfdiffusion'/'af2' on Colab (A100). See MANUAL.md §2.")

## 3 · Occlusion proxy (does it block the substrate channel?) + rim coverage

A binder only *inhibits* if it **occludes** the substrate-access channel over the di-zinc site.
`occlusion_score()` combines rim coverage (`hotspot_overlap`, the fraction of rim hotspots contacted)
with a pocket-fit term — a teaching stand-in for the pocket-volume / docking analysis in notebook 04
and the inhibition assay in notebook 05. Higher ⇒ more likely to block (**not** a guarantee:
occlusion ≠ inhibition).

In [ ]:
for b in bc[:3]:
    ov = bt.hotspot_overlap(b.contact_residues, HOTSPOTS)
    occ = bt.occlusion_score(b, HOTSPOTS, tool="mock")
    print(f"{b.design_id}: contacts {b.contact_residues} -> rim coverage = {ov}, "
          f"occlusion = {occ['occlusion']} (occludes? {occ['occludes']}) (SYNTHETIC)")
print("\nNOTE:", occ["note"])

## 4 · Specificity proxy vs a human metalloenzyme (safety counter-test)

A di-zinc-site binder that also hits a **human** Zn/metalloenzyme (carbonic anhydrase, MMPs,
glyoxalase II) is a safety liability. `offtarget_specificity()` returns a 0–1 selectivity (higher =
more NDM-1-selective). This is the in-silico mirror of the off-target-metalloenzyme **control** in the
inhibition assay (notebook 05). SYNTHETIC on the mock backend.

In [ ]:
spec = bt.offtarget_specificity(bc[0], "CA2", tool="mock")   # CA2 = human carbonic anhydrase II
print(f"{bc[0].design_id} vs human {spec['off_target']}: specificity = {spec['specificity']} "
      f"(selective? {spec['selective']}) (SYNTHETIC)")
print("NOTE:", spec["note"])

## Visualize a binder–target complex (py3Dmol)

Use this to eyeball a predicted binder–NDM-1 complex once you have a real PDB (from AF2-Multimer) —
and to confirm the binder sits **over the di-zinc substrate-access rim**, with both Zn²⁺ retained.

In [ ]:
import py3Dmol

def show_complex(pdb_path_or_str, is_path=True):
    data = open(pdb_path_or_str).read() if is_path else pdb_path_or_str
    view = py3Dmol.view(width=520, height=420)
    view.addModel(data, "pdb")
    view.setStyle({"cartoon": {"color": "spectrum"}})
    view.addStyle({"resn": "ZN"}, {"sphere": {"color": "grey", "radius": 0.6}})  # show the di-zinc site
    view.zoomTo()
    return view.show()

# Example (after a real AF2-Multimer prediction writes a complex PDB):
# show_complex("results/af2/top_complex.pdb")
print("show_complex(pdb_path) ready (renders the di-zinc ions as spheres).")

## D0 checklist
- [ ] NDM-1 accessions verified on RCSB (3SPU/4EYL are candidates); chain identified; **both Zn²⁺ present**.
- [ ] Cleaned di-zinc target (Zn preserved) + **active-site-rim hotspot list** (the substrate-channel walls, not invented, not the Zn ligands).
- [ ] One-paragraph definition of each binder/occlusion metric **with** its "does not mean" note (esp. binding ≠ inhibition).
- [ ] Reproduced mock mini-run (both paradigms) with metrics + occlusion + specificity printed and flagged SYNTHETIC.
- [ ] Problem statement with measurable success criteria (incl. an occlusion threshold) + controls; `LOG.md` entry (GPU, seed).

**Next:** `02_generate.ipynb` — the two-paradigm binder campaign at the active-site rim.